In [ ]:
!pip install sentence-transformers==0.3.0
!pip install git+https://github.com/openai/CLIP.git

# First: Get our Concreteness scores for our dataset
Start by training our estimator following: ```Estimating Word Concreteness from Contextualized Embeddings (Wartena, KONVENS 2024)```

### Data prep

In [ ]:
import pickle
import torch
from torch.cuda.amp import autocast
from transformers import RobertaTokenizer, RobertaModel
from tqdm import tqdm
import torch.nn as nn
import numpy as np
import glob
import re
import pandas as pd

# --- Configuration ---
RATINGS_FILE_PATH = 'data/Concreteness_ratings_Brysbaert_et_al_BRM.csv'
CORPORA_PATH_PATTERN = 'data/*.txt'
OUTPUT_FILE = 'averaged_embeddings_final.pkl'

BATCH_SIZE = 256

# --- Model and Device Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Loading RoBERTa model and tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaModel.from_pretrained('roberta-base', output_hidden_states=True)
model.to(device)
model.eval()

# --- 1. Load Concreteness Ratings ---
print(f"Loading concreteness ratings from {RATINGS_FILE_PATH}...")
df_ratings = pd.read_csv(RATINGS_FILE_PATH)
df_ratings.dropna(subset=['Word'], inplace=True)
target_words = set(df_ratings['Word'].str.lower())
print(f"Successfully loaded ratings for {len(target_words)} unique words.")

# --- Dictionary to hold all embeddings ---
word_embeddings_lists = {word: [] for word in target_words}

# --- Reusable function to process a batch of text (either words or sentences) ---
def process_batch(text_batch):
    if not text_batch:
        return

    with autocast(): # Use Automatic Mixed Precision
        with torch.no_grad():
            inputs = tokenizer(text_batch, return_tensors="pt", truncation=True, max_length=512, padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)

            hidden_states = outputs.hidden_states[-4:]
            last_four_layers = torch.stack(hidden_states, dim=0)
            avg_last_four = torch.mean(last_four_layers, dim=0)

            for i, text in enumerate(text_batch):
                # Find which target words are in this specific text
                # For default embeddings, text is the word itself.
                # For corpus sentences, text is the sentence.
                words_in_text = target_words.intersection(re.findall(r'\b\w+\b', text.lower()))

                for word in words_in_text:
                    word_token_ids = tokenizer.encode(word, add_special_tokens=False)
                    input_ids_list = inputs['input_ids'][i].tolist()
                    for j in range(len(input_ids_list) - len(word_token_ids) + 1):
                        if input_ids_list[j:j+len(word_token_ids)] == word_token_ids:
                            token_embeddings = avg_last_four[i, j:j+len(word_token_ids)]
                            word_embedding = torch.mean(token_embeddings, dim=0)
                            word_embeddings_lists[word].append(word_embedding.cpu())
                            break

# --- 2. Generate Default Embeddings ---
print("\n--- Step 2: Generating default embeddings for all words ---")
word_list = list(target_words)
for i in tqdm(range(0, len(word_list), BATCH_SIZE), desc="Generating default embeddings"):
    batch = word_list[i:i + BATCH_SIZE]
    process_batch(batch)

# --- 3. Find Contextual Embeddings in Text Corpora ---
print("\n--- Step 3: Finding contextual embeddings in corpora ---")
corpus_files = glob.glob(CORPORA_PATH_PATTERN)
print(f"Found {len(corpus_files)} corpus files.")

for file_path in corpus_files:
    print(f"\nProcessing file: {file_path}")
    sentence_batch = []
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for sentence in tqdm(f, desc=f"Reading {file_path.split('/')[-1]}"):
            sentence = sentence.strip()
            if sentence:
                sentence_batch.append(sentence)
            if len(sentence_batch) == BATCH_SIZE:
                process_batch(sentence_batch)
                sentence_batch = []
    process_batch(sentence_batch) # Process final batch

print("\nFinished processing all corpora.")

# --- 4. Average the collected embeddings and prepare final dataset ---
print("\n--- Step 4: Averaging all collected embeddings ---")
final_data = []
words_with_embeddings = 0
for word, embeddings_list in tqdm(word_embeddings_lists.items(), desc="Averaging"):
    if embeddings_list:
        stacked_embeddings = torch.stack(embeddings_list, dim=0)
        avg_embedding = torch.mean(stacked_embeddings, dim=0)
        conc_score = df_ratings.loc[df_ratings['Word'].str.lower() == word, 'Conc.M'].iloc[0]
        final_data.append({
            'word': word,
            'embedding': avg_embedding.numpy(),
            'concreteness': conc_score
        })
        words_with_embeddings += 1

print(f"\nSuccessfully generated averaged embeddings for {words_with_embeddings}/{len(target_words)} words.")

# --- 5. Save the final data ---
print(f"Saving the final dataset to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'wb') as f:
    pickle.dump(final_data, f)

print("\n--- Process Complete ---")
print(f"Your data is ready in '{OUTPUT_FILE}'.")

### SVR

In [ ]:
import pickle
import numpy as np
# 1. Import SVR from cuML instead of sklearn. Multiple orders of maginute quicker.
from cuml.svm import SVR
# Import cupy for GPU array handling
import cupy as cp
from sklearn.model_selection import train_test_split
import joblib

# --- Configuration ---
INPUT_DATA_FILE = 'averaged_embeddings_final.pkl'
MODEL_SAVE_PATH = 'concreteness_svr_model_gpu.joblib' # New model name
TEST_SPLIT_SIZE = 0.05

# --- 1. Load Data (Remains the same) ---
print(f"Loading data from {INPUT_DATA_FILE}...")
with open(INPUT_DATA_FILE, 'rb') as f:
    data = pickle.load(f)

X = np.array([item['embedding'] for item in data])
y = np.array([item['concreteness'] for item in data])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT_SIZE, random_state=42
)

# 2. Convert data to CuPy arrays for the GPU
print("Moving data to GPU memory...")
X_train_gpu = cp.asarray(X_train)
y_train_gpu = cp.asarray(y_train)
X_test_gpu = cp.asarray(X_test)
y_test_gpu = cp.asarray(y_test)

print(f"Data loaded. Training on {len(X_train)} samples.")

# --- 3. Define and Train the cuML SVR Model ---
print("\nInitializing GPU-accelerated SVR model (kernel='poly')...")
# Use the SVR object from cuML
svr_model_gpu = SVR(kernel='poly', cache_size=1000, verbose=True)

print("Training SVR model on GPU... (This should be very fast)")
svr_model_gpu.fit(X_train_gpu, y_train_gpu)
print("Training finished.")

# --- 4. Evaluate the Model ---
print("\nEvaluating model performance on the test set...")
r2 = svr_model_gpu.score(X_test_gpu, y_test_gpu)
print(f"Model R^2 score on test set: {r2:.4f}")

# --- 5. Save the Trained Model ---
joblib.dump(svr_model_gpu, MODEL_SAVE_PATH)
print(f"\nGPU SVR model saved to {MODEL_SAVE_PATH}")

In [ ]:
import torch
from transformers import RobertaTokenizerFast, RobertaModel
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import numpy as np
import joblib

# --- Configuration ---
SVR_MODEL_PATH = 'concreteness_svr_model_gpu.joblib'
# Path to your CONCRETEXT test file
TEST_SET_PATH = 'CONcreTEXT_test_EN.tsv'
BATCH_SIZE = 32

# --- Setup Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- 1. Load Models ---
print("Loading RoBERTa model and FAST tokenizer...")
tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')
roberta_model = RobertaModel.from_pretrained('roberta-base', output_hidden_states=True)
roberta_model.to(device)
roberta_model.eval()

print(f"Loading trained SVR model from {SVR_MODEL_PATH}...")
svr_model = joblib.load(SVR_MODEL_PATH)

# --- 2. Load Test Data ---
try:
    print(f"Loading test data from {TEST_SET_PATH}...")
    # Use sep='\t' to correctly read the tab-separated file
    test_df = pd.read_csv(TEST_SET_PATH, sep='\t')
    test_df.columns = test_df.columns.str.strip()
    print(f"Found {len(test_df)} samples in the test set.")
except FileNotFoundError:
    print(f"ERROR: Test file not found at {TEST_SET_PATH}. Please update the path.")
    exit()

# --- 3. Batched Prediction and Evaluation Loop ---
gold_scores = []
predicted_scores_raw = []

for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Evaluating Test Set with SVR"):
    batch_df = test_df.iloc[i:i + BATCH_SIZE]

    contexts = batch_df['TEXT'].tolist()
    targets = batch_df['TARGET'].tolist()

    with torch.no_grad():
        inputs = tokenizer(contexts, return_tensors="pt", truncation=True, max_length=512, padding=True)

        token_indices = []
        for j, (text, target) in enumerate(zip(contexts, targets)):
            target_str = str(target)
            start_char = text.lower().find(target_str.lower())

            if start_char == -1:
                token_indices.append(None)
                continue
            end_char = start_char + len(target_str)

            start_token = inputs.char_to_token(j, start_char)
            end_token = inputs.char_to_token(j, end_char - 1)

            if start_token is None or end_token is None:
                token_indices.append(None)
            else:
                token_indices.append((start_token, end_token))

        gpu_inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = roberta_model(**gpu_inputs)
        avg_last_four = torch.mean(torch.stack(outputs.hidden_states[-4:], dim=0), dim=0)

        pooled_embeddings = []
        valid_batch_indices = []
        for j, indices in enumerate(token_indices):
            if indices:
                start_token, end_token = indices
                token_embeddings = avg_last_four[j, start_token : end_token + 1]
                concept_embedding = torch.mean(token_embeddings, dim=0)
                pooled_embeddings.append(concept_embedding)
                valid_batch_indices.append(j)

        if pooled_embeddings:
            # Convert embeddings to CPU numpy array for SVR prediction
            batch_numpy = torch.stack(pooled_embeddings).cpu().numpy()

            # Get predictions from the SVR model
            predictions = svr_model.predict(batch_numpy)

            if not isinstance(predictions, np.ndarray):
                predictions = [predictions]
            else:
                predictions = predictions.tolist()

            # Align predictions with their gold scores
            for k, pred_score in enumerate(predictions):
                original_index = valid_batch_indices[k]
                predicted_scores_raw.append(pred_score)
                gold_scores.append(batch_df.iloc[original_index]['AVG'])

# --- 4. Rescale Predictions and Calculate Correlation ---
print("\n--- SVR Model Evaluation Results ---")
if not predicted_scores_raw:
    print("Could not make any successful predictions on the test set.")
else:
    # Rescale from the model's native [1, 5] scale to the test set's [1, 7] scale
    rescaled_scores = [1 + (score - 1) * 1.5 for score in predicted_scores_raw]

    # Calculate correlation coefficients
    pearson_corr, _ = pearsonr(rescaled_scores, gold_scores)
    spearman_corr, _ = spearmanr(rescaled_scores, gold_scores)

    print(f"Successfully processed and compared {len(rescaled_scores)} samples.")
    print(f"Pearson Correlation Coefficient: {pearson_corr:.4f}")
    print(f"Spearman Correlation Coefficient: {spearman_corr:.4f}")

# Extracting Wikipedia

Create popularity map

In [ ]:
# Download the file from the URL
!wget https://dumps.wikimedia.org/other/pageview_complete/2025/2025-06/pageviews-20250630-user.bz2

# Decompress the file
!bzip2 -d pageviews-20250630-user.bz2

In [ ]:
import pickle
from tqdm import tqdm

# --- Configuration ---
PAGEVIEW_FILE = 'pageviews-20250630-user'
OUTPUT_MAP_FILE = 'popularity_map.pkl'

# --- Main Script ---
popularity_map = {}
print(f"Processing pageview file: {PAGEVIEW_FILE}...")

try:
    total_lines = sum(1 for line in open(PAGEVIEW_FILE, 'r', encoding='utf-8'))
except FileNotFoundError:
    print(f"ERROR: Pageview file not found at '{PAGEVIEW_FILE}'. Please ensure it is downloaded and unzipped.")
    exit()

with open(PAGEVIEW_FILE, 'r', encoding='utf-8') as f:
    for line in tqdm(f, total=total_lines, desc="Building Popularity Map"):
        parts = line.strip().split(' ')

        if len(parts) >= 4 and parts[0].startswith('en.'):
            try:
                page_id_str = parts[-4]
                view_count = int(parts[-2])

                if page_id_str == 'null':
                    continue

                page_id = int(page_id_str)

                # Add to the existing count, or initialize it if it's the first time we see this ID.
                popularity_map[page_id] = popularity_map.get(page_id, 0) + view_count
            except (ValueError, IndexError):
                print("Error in parsing line")
                print(line)
                print(parts)
                break

print(f"\nCreated map with {len(popularity_map)} entries.")

with open(OUTPUT_MAP_FILE, 'wb') as f:
    pickle.dump(popularity_map, f)

print(f"Popularity map saved to {OUTPUT_MAP_FILE}")

### Get Wikipedia titles and estimate concreteness score

In [ ]:
# Step 1: Install Miniconda, a lightweight environment manager
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -bfp /usr/local
!rm Miniconda3-latest-Linux-x86_64.sh

# Step 2: Create a self-contained Python 3.10 environment
!conda create -n py310 python=3.10 -y

# Step 3: Install a compatible version of wikiextractor in the new environment
# We use v3.0.6 as it is known to work well with Python 3.10
!conda run -n py310 pip install wikiextractor==3.0.6

# Step 4: Run wikiextractor using the Python 3.10 environment
# Ensure your input/output file paths are correct for your Drive setup
!conda run -n py310 python -m wikiextractor.WikiExtractor --processes $(nproc) -o extracted enwiki-latest-pages-articles.xml.bz2 2> wikiextractor.log
!echo "Finished"

Get the relevant Wikipedia pages

In [ ]:
import os
import re
import csv
import glob
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
import nltk

# --- Configuration ---
INPUT_DIR = 'extracted'
OUTPUT_CSV_PATH = 'wiki_data_with_ids.csv' # New output filename
NLTK_DATA_PATH = os.path.join(os.getcwd(), 'nltk_data')

# --- Download NLTK data ---
os.makedirs(NLTK_DATA_PATH, exist_ok=True)
nltk.data.path.append(NLTK_DATA_PATH)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', download_dir=NLTK_DATA_PATH)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', download_dir=NLTK_DATA_PATH)

# --- Initializer for worker processes ---
def init_worker(nltk_path):
    nltk.data.path.append(nltk_path)

# --- Worker Function ---
def process_file(file_path):
    results = []
    disambiguation_phrases = ['may refer to:', 'is a set of', 'can refer to', 'is a list of']

    # This new regex now captures the 'id' attribute as the first group
    article_regex = re.compile(r'<doc id="(.*?)".*?title="(.*?)">(.*?)<\/doc>', re.DOTALL)

    with open(file_path, 'r', encoding='utf-8') as infile:
        content = infile.read()
        articles = article_regex.findall(content)

        for page_id, title, text in articles:
            # Skip sub-pages and CamelCase titles
            if '/' in title or (' ' not in title and re.search(r'[a-z][A-Z]', title)):
                continue

            clean_text = text.replace('\n', ' ').strip()

            if len(clean_text) < len(title) + 20:
                continue

            try:
                sentences = nltk.sent_tokenize(clean_text)
            except Exception:
                continue

            if not sentences:
                continue

            first_sentence = sentences[0]

            if len(first_sentence) < 25 or any(phrase in first_sentence.lower() for phrase in disambiguation_phrases):
                continue

            if title.lower() not in first_sentence.lower():
                continue

            # Append all three pieces of data
            results.append((page_id, title, first_sentence))

    return results

# --- Main script execution ---
if __name__ == '__main__':
    input_files = glob.glob(os.path.join(INPUT_DIR, '**', 'wiki_*'), recursive=True)

    if not input_files:
        print(f"ERROR: No processed files found in the '{INPUT_DIR}' directory.")
    else:
        print(f"Found {len(input_files)} files to process. Extracting Page ID, Title, and Context...")

        with open(OUTPUT_CSV_PATH, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            # Update header for the new column
            writer.writerow(['Page_ID', 'Title', 'Context'])

            with Pool(processes=cpu_count(), initializer=init_worker, initargs=(NLTK_DATA_PATH,)) as pool:
                for results_list in tqdm(pool.imap_unordered(process_file, input_files), total=len(input_files)):
                    if results_list:
                        writer.writerows(results_list)

        print(f"\nProcessing complete. Your final data has been saved to '{OUTPUT_CSV_PATH}'")

Create final, unified CSV with predicted concreteness scores

In [ ]:
import torch
from transformers import RobertaTokenizerFast, RobertaModel
import joblib
import pandas as pd
from tqdm import tqdm
import time
import csv
import pickle

# --- Configuration ---
SVR_MODEL_PATH = 'concreteness_svr_model_gpu.joblib'
INPUT_FILE_PATH = 'wiki_data_with_ids.csv'
POPULARITY_MAP_PATH = 'popularity_map.pkl'
OUTPUT_FILE_PATH = 'final_scores_with_popularity.csv'
BATCH_SIZE = 128

# --- Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Load Models ---
print("Loading RoBERTa model and FAST tokenizer...")
tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')
roberta_model = RobertaModel.from_pretrained('roberta-base', output_hidden_states=True)
roberta_model.to(device)
roberta_model.eval()

print(f"Loading trained SVR model from {SVR_MODEL_PATH}...")
svr_model = joblib.load(SVR_MODEL_PATH)

with open(POPULARITY_MAP_PATH, 'rb') as f:
    popularity_map = pickle.load(f)
print(f"Loaded popularity map with {len(popularity_map)} entries.")

# --- Load Input Data ---
try:
    print(f"Loading input data from {INPUT_FILE_PATH}...")
    df = pd.read_csv(INPUT_FILE_PATH)
    df['Page_ID'] = pd.to_numeric(df['Page_ID'], errors='coerce').astype('Int64')
    print(f"Found {len(df)} concepts to process.")
except FileNotFoundError:
    print(f"ERROR: Input file not found at {INPUT_FILE_PATH}.")
    exit()

# --- Prediction Function ---
def predict_batch_final(concept_titles, context_sentences):
    with torch.no_grad():
        inputs = tokenizer(context_sentences, return_tensors="pt", truncation=True, max_length=512, padding=True)
        gpu_inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = roberta_model(**gpu_inputs)
        avg_last_four = torch.mean(torch.stack(outputs.hidden_states[-4:], dim=0), dim=0)

        pooled_embeddings = []
        valid_indices = []
        for i, title in enumerate(concept_titles):
            start_char = context_sentences[i].lower().find(title.lower())
            if start_char == -1: continue
            end_char = start_char + len(title)

            start_token = inputs.char_to_token(i, start_char)
            end_token = inputs.char_to_token(i, end_char - 1)

            if start_token is not None and end_token is not None:
                token_embeddings = avg_last_four[i, start_token : end_token + 1]
                concept_embedding = torch.mean(token_embeddings, dim=0)
                pooled_embeddings.append(concept_embedding)
                valid_indices.append(i)

        if not pooled_embeddings:
            return [], []

        batch_numpy = torch.stack(pooled_embeddings).cpu().numpy()
        predictions = svr_model.predict(batch_numpy).tolist()

        return predictions, valid_indices

# --- Main Processing Loop ---
print(f"Processing concepts and saving results to {OUTPUT_FILE_PATH}...")
start_time = time.time()

with open(OUTPUT_FILE_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    # Update header for the new column
    writer.writerow(['Title', 'Concreteness_Score', 'Popularity_Score'])

    for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Processing Batches"):
        batch_df = df.iloc[i:i + BATCH_SIZE]
        titles_batch = batch_df['Title'].tolist()
        contexts_batch = batch_df['Context'].tolist()

        predictions, valid_indices = predict_batch_final(titles_batch, contexts_batch)

        # Get the original dataframe rows for which predictions were successful
        valid_df = batch_df.iloc[valid_indices]

        # Look up popularity scores using the Page_ID
        # .get(id, 0) returns 0 if an ID is not in the map
        popularity_scores = [popularity_map.get(page_id, 0) for page_id in valid_df['Page_ID']]

        rows_to_write = [
            (valid_df.iloc[k]['Title'], predictions[k], popularity_scores[k])
            for k in range(len(predictions))
        ]

        if rows_to_write:
            writer.writerows(rows_to_write)


end_time = time.time()
total_time = end_time - start_time
print("\n--- Processing Complete ---")
print(f"Finished processing all concepts in {total_time/60:.2f} minutes (or {total_time:.2f} seconds).")
print(f"Your final results have been saved to {OUTPUT_FILE_PATH}")

# Create final, unified file to query.
### This requires a lot of RAM and VRAM

In [ ]:
import torch
import clip
from pathlib import Path
import re
from tqdm import tqdm
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import os

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 512

# 1. INPUT
CONCRETENESS_FILE = Path("final_scores_with_popularity.csv")

# 2. OUTPUT
OUTPUT_INDEX_FILE = Path("dual_embedding_index_with_scores.pt")

# 3. Embedding Model
QWEN_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"


def create_structured_index(concreteness_csv_path, output_path):
    """
    Encodes concepts using both models and saves the index by streaming
    embeddings to disk to conserve RAM.
    """
    if not concreteness_csv_path.exists():
        print(f"Error: Input file not found at '{concreteness_csv_path}'")
        return

    print(f"Using device: {DEVICE}")
    clip_model, _ = clip.load('ViT-B/32', device=DEVICE)
    qwen_model = SentenceTransformer(QWEN_MODEL_NAME, device=DEVICE)

    print(f"Reading data from {concreteness_csv_path}...")
    df = pd.read_csv(concreteness_csv_path)
    concepts = df['Title'].tolist()
    conc_scores = df['Concreteness_Score'].tolist()
    pop_scores = df['Popularity_Score'].tolist()

    num_concepts = len(concepts)
    # Get embedding dimensions
    clip_dim = 512 # For ViT-B/32
    qwen_dim = qwen_model.get_sentence_embedding_dimension()

    ## --- Create memory-mapped files on disk ---
    # These files will act as our RAM, storing the embeddings
    temp_clip_path = "temp_clip_embeddings.npy"
    temp_qwen_path = "temp_qwen_embeddings.npy"

    # Create empty arrays on disk with the full final shape
    clip_mmap = np.memmap(temp_clip_path, dtype='float32', mode='w+', shape=(num_concepts, clip_dim))
    qwen_mmap = np.memmap(temp_qwen_path, dtype='float32', mode='w+', shape=(num_concepts, qwen_dim))

    print(f"Starting encoding process for {num_concepts} concepts...")
    for i in tqdm(range(0, len(concepts), BATCH_SIZE), desc="Encoding Batches"):
        batch_of_concepts = concepts[i:i + BATCH_SIZE]
        start_index = i
        end_index = i + len(batch_of_concepts)

        try:
            # --- CLIP Encoding ---
            clip_token_tensor = clip.tokenize(batch_of_concepts, truncate=True).to(DEVICE)
            with torch.no_grad(), torch.autocast(device_type="cuda"):
                clip_embedding_tensor = clip_model.encode_text(clip_token_tensor).cpu()

                ## --- Step 2 - Write batch directly to disk ---
                clip_mmap[start_index:end_index] = clip_embedding_tensor.numpy()

            # --- Qwen Encoding ---
            with torch.no_grad(), torch.autocast(device_type="cuda"):
                qwen_embedding_tensor = qwen_model.encode(
                    batch_of_concepts, convert_to_tensor=True, device=DEVICE
                ).cpu()

                ## --- Step 2 - Write batch directly to disk ---
                qwen_mmap[start_index:end_index] = qwen_embedding_tensor.numpy()

        except Exception as e:
            print(f"\nWarning: Skipping a batch due to an error: {e}")
            continue

    # Flush memory maps to ensure all data is written to disk
    clip_mmap.flush()
    qwen_mmap.flush()

    print("\nProcessing complete. Assembling final data from disk...")

    ## --- Step 3 - Load embeddings from disk files for final processing ---
    final_clip_embeddings_tensor = torch.from_numpy(np.load(temp_clip_path))
    final_clip_embeddings_tensor /= final_clip_embeddings_tensor.norm(dim=-1, keepdim=True)

    final_qwen_embeddings_tensor = torch.from_numpy(np.load(temp_qwen_path))
    final_qwen_embeddings_tensor /= final_qwen_embeddings_tensor.norm(dim=-1, keepdim=True)

    # Create a single dictionary to hold all data
    final_data = {
        'concepts': concepts,
        'conc_scores': torch.tensor(conc_scores, dtype=torch.float16),
        'pop_scores': torch.tensor(pop_scores, dtype=torch.float16),
        'clip_embeddings': final_clip_embeddings_tensor.to(torch.float16),
        'qwen_embeddings': final_qwen_embeddings_tensor.to(torch.float16)
    }

    print(f"Saving final structured index to {output_path}...")
    torch.save(final_data, output_path)


    print("Indexing complete!")


create_structured_index(CONCRETENESS_FILE, OUTPUT_INDEX_FILE)

In case of a crash due to RAM/VRAM constraints:

In [ ]:
import torch
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHUNK_SIZE = 100000

# --- File Paths ---
CONCRETENESS_FILE = Path("final_scores_with_popularity.csv")
OUTPUT_INDEX_FILE = Path("dual_embedding_index_with_scores.pt")

# Point to the existing (but header-corrupted) files
CLIP_TEMP_PATH = "temp_clip_embeddings.npy"
QWEN_TEMP_PATH = "temp_qwen_embeddings.npy"
CLIP_PROGRESS_FILE = "clip_progress.txt" # To clean up
QWEN_PROGRESS_FILE = "qwen_progress.txt" # To clean up


def manual_recover_and_assemble():
    """
    Recovers data from raw .npy files with broken headers and assembles the final index.
    """
    print("--- Starting MANUAL RECOVERY and Final Assembly ---")

    # --- Step 1: Load metadata to determine the expected shape ---
    print(f"Reading concepts and scores from {CONCRETENESS_FILE}...")
    df = pd.read_csv(CONCRETENESS_FILE)
    concepts = df['Title'].tolist()
    conc_scores = torch.tensor(df['Concreteness_Score'].tolist(), dtype=torch.float16)
    pop_scores = torch.tensor(df['Popularity_Score'].tolist(), dtype=torch.float16)
    num_concepts = len(concepts)

    # Define the expected dimensions
    clip_dim = 512
    qwen_dim = 1024 # For Qwen3-Embedding-0.6B

    # --- Step 2: Manually load raw data using np.fromfile ---
    print("Bypassing headers and reading raw binary data from .npy files...")

    # Read the flat binary data
    clip_data_flat = np.fromfile(CLIP_TEMP_PATH, dtype=np.float32)
    qwen_data_flat = np.fromfile(QWEN_TEMP_PATH, dtype=np.float32)

    # Reshape the flat data back into the correct 2D array structure
    clip_embeddings_numpy = clip_data_flat.reshape(num_concepts, clip_dim)
    qwen_embeddings_numpy = qwen_data_flat.reshape(num_concepts, qwen_dim)

    print("Raw data successfully recovered and reshaped!")

    # --- Step 3: Create final empty tensors ---
    final_clip_embeddings = torch.empty(clip_embeddings_numpy.shape, dtype=torch.float16)
    final_qwen_embeddings = torch.empty(qwen_embeddings_numpy.shape, dtype=torch.float16)

    # --- Step 4: Process the recovered arrays in chunks ---
    print("Normalizing embeddings in memory-safe chunks...")
    for i in tqdm(range(0, num_concepts, CHUNK_SIZE), desc="Processing Chunks"):
        end_index = min(i + CHUNK_SIZE, num_concepts)

        clip_chunk = torch.from_numpy(clip_embeddings_numpy[i:end_index]).to(DEVICE)
        clip_chunk /= clip_chunk.norm(dim=-1, keepdim=True)
        final_clip_embeddings[i:end_index] = clip_chunk.cpu().to(torch.float16)

        qwen_chunk = torch.from_numpy(qwen_embeddings_numpy[i:end_index]).to(DEVICE)
        qwen_chunk /= qwen_chunk.norm(dim=-1, keepdim=True)
        final_qwen_embeddings[i:end_index] = qwen_chunk.cpu().to(torch.float16)

    # --- Step 5: Assemble and Save ---
    print("Assembling and saving final data structure...")
    final_data = {
        'concepts': concepts,
        'conc_scores': conc_scores,
        'pop_scores': pop_scores,
        'clip_embeddings': final_clip_embeddings,
        'qwen_embeddings': final_qwen_embeddings
    }
    torch.save(final_data, OUTPUT_INDEX_FILE)

    # --- Step 6: Clean up temporary files ---
    # print("Cleaning up temporary and progress files...")
    # os.remove(CLIP_TEMP_PATH)
    # os.remove(QWEN_TEMP_PATH)
    # if os.path.exists(CLIP_PROGRESS_FILE): os.remove(CLIP_PROGRESS_FILE)
    # if os.path.exists(QWEN_PROGRESS_FILE): os.remove(QWEN_PROGRESS_FILE)

    print("Recovery and final assembly complete!")

# Run the manual recovery process
manual_recover_and_assemble()